# Análisis de Política Monetaria Mexicana
## Series de Tiempo con Python

### ¿Qué hace este proyecto?
Este proyecto analiza la evolución de la inflación en México, su relación con la tasa de interés y el tipo de cambio, así como la respuesta del Banco de México con su tasa objetivo mediante la política monetaria. De igual forma se hace una proyección de inflación con modelos de series de tiempo.

### Datos
Se usan los datos del Sistema de Información Económica (SIE) del Banco de México mediante su API, desde enero de 2008 hasta la fecha más reciente disponible.

| Serie SIE | Indicador |
|---|---|
| SP30578 | Inflación general anual (INPC) |
| SP74662 | Inflación subyacente anual |
| SF61745 | Tasa objetivo Banxico |
| SF43718 | Tipo de cambio USD/MXN (FIX) |

### ¿Cómo ejecutarlo?
1. Abrir **Anaconda Prompt**
2. Activar el environment: `conda activate banxico`
3. Abrir Jupyter: `jupyter notebook`
4. Abrir este archivo y ejecutar cada celda en orden descendente

### Hallazgos principales
1. **Diagnóstico de datos** — La inflación mexicana 2008-2026 vivió tres shocks brutales: la crisis financiera 2008-2009, la pandemia en 2020, y el ciclo inflacionario post-pandemia 2021-2022.
2. **Causalidad** — La inflación Granger-causa a la tasa Banxico (lags 1-2, p<0.05), pero no al revés. El Banxico reacciona a la inflación con 1-2 meses de rezago — opera una regla de reacción, no de anticipación.
3. **Modelos** — ARIMA(0,1,1) proyecta línea plana, insuficiente. Prophet captura tendencia pero no shocks exógenos. El VAR con 3 lags es el más adecuado porque captura las interdependencias entre variables.
4. **Proyección** — El VAR anticipa inflación convergiendo hacia 4.25% y la tasa bajando a ~6% para finales de 2026 — ciclo de recortes sostenido pero inflación aún por encima de la meta del 3%.

---
## PARTE 1 — Instalación de librerías

Una **librería** es como un plugin que le agrega funcionalidades extra a Python base.  
Solo necesitas correr esta celda **una vez** por environment.

| Librería | ¿Para qué la usamos? |
|---|---|
| `sie-banxico` | Conectarse a la API del Banxico |
| `pandas` | Manipular tablas de datos (como Excel pero en código) |
| `plotly` | Gráficas interactivas |
| `statsmodels` | Modelos estadísticos y pruebas de series de tiempo |
| `pmdarima` | Encontrar el mejor modelo ARIMA automáticamente |
| `prophet` | Modelo de proyección de Meta/Facebook |

In [ ]:
# El signo ! le dice a Jupyter que ejecute este comando en la terminal
# Solo hay que correr esta celda UNA VEZ
import sys
!{sys.executable} -m pip install sie-banxico pandas plotly statsmodels pmdarima prophet -q
print('✅ Librerías instaladas correctamente')

---
## PARTE 2 — Importación de librerías y configuración

Instalar = descargar la librería al equipo  
Importar = activarla para usarla en este notebook  
Son dos pasos separados — la celda anterior instala, esta importa.

In [ ]:
# Librería para conectarse a la API de Banxico
# Nota: el nombre del paquete es 'sie-banxico' pero se importa como 'sie_banxico'
# (los guiones se convierten en guiones bajos al importar en Python)
from sie_banxico import SIEBanxico

# Pandas: manejo de tablas de datos (DataFrames)
import pandas as pd

# NumPy: operaciones matemáticas y numéricas
import numpy as np

# Plotly: gráficas interactivas
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Statsmodels: pruebas estadísticas y modelos de series de tiempo
from statsmodels.tsa.stattools import adfuller, acf, pacf, grangercausalitytests
from statsmodels.tsa.api import VAR

# Pmdarima: encuentra el mejor modelo ARIMA automáticamente
import pmdarima as pm

# Prophet: modelo de proyección de Meta/Facebook
from prophet import Prophet

# Ocultamos advertencias para que el output se vea más limpio
import warnings
warnings.filterwarnings('ignore')

print('✅ Todo importado correctamente, listo para arrancar')

---
## PARTE 3 — Conexión a la API del Banxico y descarga de datos

Una **API** (Application Programming Interface) es el intermediario entre nuestro código y los servidores de Banxico.  
En lugar de descargar archivos manualmente, le pedimos los datos directamente desde Python.

El **token** es nuestra credencial de acceso — se obtiene en https://www.banxico.org.mx/SieAPIRest/service/v1/token  
Cada serie de datos tiene un **código único** en el SIE (similar al SKU de un producto).

**¿Por qué desde 2008?**  
La tasa objetivo de Banxico existe formalmente desde enero de 2008. Antes usaba otro instrumento diferente.  
Además capturamos tres ciclos económicos completos: crisis financiera 2008-2009, pandemia 2020, y ciclo inflacionario 2021-2022.

In [ ]:
# ── Token de acceso ──────────────────────────────────────────
# Sustituye el texto entre comillas por tu token real de 64 caracteres
TOKEN = 'AQUI_VA_TU_TOKEN_DE_64_CARACTERES'

# Verificación del formato del token
if len(TOKEN) == 64:
    print('✅ Token con longitud correcta (64 caracteres)')
elif TOKEN == 'AQUI_VA_TU_TOKEN_DE_64_CARACTERES':
    print('⚠️  No has sustituido el token. Reemplaza el texto de arriba con tu token real.')
else:
    print(f'⚠️  Token con longitud inesperada ({len(TOKEN)} caracteres). Verifica que lo copiaste completo.')

In [ ]:
# ── Códigos de las series que vamos a descargar ──────────────
# Estos códigos se encontraron verificando el SIE de Banxico directamente
# para asegurarnos que son variaciones anuales en porcentaje (no índices)
SERIES = {
    'SP30578': 'Inflación general anual',     # Variación anual del INPC
    'SP74662': 'Inflación subyacente anual',  # Inflación sin energía ni alimentos volátiles
    'SF61745': 'Tasa objetivo Banxico',       # Instrumento principal de política monetaria
    'SF43718': 'Tipo de cambio USD/MXN (FIX)' # Termómetro de confianza en la economía
}

# Periodo de análisis
FECHA_INICIO = '2008-01-01'  # Desde que existe formalmente la tasa objetivo
FECHA_FIN    = '2026-04-30'  # Fecha más reciente disponible

# Creamos la conexión con la API del Banxico
api = SIEBanxico(
    token=TOKEN,
    id_series=list(SERIES.keys())
)

# Descargamos los datos
# La API regresa un JSON — un diccionario anidado con los datos de cada serie
respuesta = api.get_timeseries_range(init_date=FECHA_INICIO, end_date=FECHA_FIN)
series_raw = respuesta['bmx']['series']

print('✅ Datos descargados exitosamente')
print(f'📦 Series recibidas: {len(series_raw)}')
for s in series_raw:
    print(f"   • {s['idSerie']}: {len(s['datos'])} observaciones")

---
## PARTE 4 — Construcción del DataFrame

Un **DataFrame** es como una hoja de Excel en Python: filas = observaciones, columnas = variables.

**Lección aprendida:** Banxico no siempre regresa las series en el mismo orden en que las pedimos.
Por eso emparejamos cada serie con su nombre usando su **ID único**, no por posición.

In [ ]:
# Diccionario para emparejar ID → nombre de columna
# Usamos el ID de cada serie para buscar su nombre — no dependemos del orden de llegada
NOMBRES = {
    'SP30578': 'Inflación general anual',
    'SP74662': 'Inflación subyacente anual',
    'SF61745': 'Tasa objetivo Banxico',
    'SF43718': 'Tipo de cambio USD/MXN (FIX)'
}

# Convertimos cada serie de JSON a DataFrame y las guardamos en una lista
dfs = []
for serie in series_raw:
    id_serie = serie['idSerie']          # ID de esta serie (ej. 'SP30578')
    nombre   = NOMBRES[id_serie]         # Nombre legible (ej. 'Inflación general anual')

    df_serie = pd.DataFrame(serie['datos'])  # Los datos vienen como lista de {fecha, dato}

    # La fecha viene como texto ('31/01/2018') → la convertimos a formato fecha real
    df_serie['fecha'] = pd.to_datetime(df_serie['fecha'], format='%d/%m/%Y')

    # El valor viene como texto ('5.55') → lo convertimos a número
    # errors='coerce' convierte valores raros en NaN en lugar de romper el código
    df_serie['dato'] = pd.to_numeric(df_serie['dato'], errors='coerce')

    # Renombramos la columna y establecemos la fecha como índice
    df_serie = df_serie.rename(columns={'dato': nombre}).set_index('fecha')
    dfs.append(df_serie)

# Unimos todos los DataFrames por fecha (como un VLOOKUP/BUSCARV en Excel)
df = dfs[0].join(dfs[1:], how='outer').sort_index()

# Verificación: los valores deben tener sentido económico
print('✅ DataFrame construido')
print('\n🔍 Verificación de valores más recientes:')
for col in df.columns:
    ultimo = df[col].dropna().iloc[-1]
    print(f'   {col}: {ultimo:.2f}')

---
## PARTE 5 — Exploración y limpieza de datos (EDA)

Antes de analizar o modelar, siempre hay que **conocer** los datos.

**¿Por qué hay valores faltantes (NaN)?**  
Las series tienen frecuencias distintas: inflación es mensual (1 dato/mes), tipo de cambio es diario.
Al unirlas en una tabla, los días sin dato de inflación quedan como NaN — no son errores, son huecos esperados.

**Solución:** resamplear (cambiar la frecuencia temporal de una serie) todo a frecuencia mensual — promediamos los valores diarios por mes.

In [ ]:
# Vista general del DataFrame original (frecuencia mixta)
print(f'📐 DataFrame original: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'📅 Periodo: {df.index.min().date()} → {df.index.max().date()}')
print(f'\n❓ Valores faltantes por columna:')
print(df.isnull().sum())
print('\n🔍 Primeras 5 filas:')
df.head()

In [ ]:
# Resampleamos a frecuencia mensual
# 'ME' = Month End (último día de cada mes)
# .mean() promedia todos los valores diarios de ese mes
df_mensual = df.resample('ME').mean().dropna(how='all')

print(f'✅ DataFrame mensual: {df_mensual.shape[0]} meses × {df_mensual.shape[1]} variables')
print('\n📊 Estadísticas descriptivas:')
df_mensual.describe().round(2)

In [ ]:
# Vista del DataFrame mensual — ya sin huecos problemáticos
print('🔍 Primeros 12 meses:')
df_mensual.round(2).head(12)

---
## PARTE 6 — Visualizaciones

Usamos **Plotly** para gráficas interactivas — puedes hacer zoom, ver valores al pasar el mouse, y más.  
Las gráficas cuentan la historia de la política monetaria mejor que cualquier tabla.

In [ ]:
# ── Gráfica 1: Evolución de la inflación ─────────────────────
# La meta de Banxico es 3% con una banda de tolerancia de ±1 punto porcentual (2%-4%)
# La inflación subyacente excluye energía y alimentos volátiles — es la señal más limpia

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=df_mensual.index, y=df_mensual['Inflación general anual'],
    name='Inflación general', line=dict(color='#e63946', width=2.5)
))

fig1.add_trace(go.Scatter(
    x=df_mensual.index, y=df_mensual['Inflación subyacente anual'],
    name='Inflación subyacente', line=dict(color='#f4a261', width=2, dash='dot')
))

# Línea de la meta y banda de tolerancia de Banxico
fig1.add_hline(y=3, line_dash='dash', line_color='green',
    annotation_text='Meta Banxico: 3%', annotation_position='bottom right')
fig1.add_hrect(y0=2, y1=4, fillcolor='green', opacity=0.08,
    annotation_text='Banda de tolerancia (±1pp)', annotation_position='top left')

fig1.update_layout(
    title='<b>Inflación en México (2008–2026)</b><br><sup>Variación anual del INPC — La subyacente es la señal, la general es la señal + ruido externo</sup>',
    xaxis_title='Fecha', yaxis_title='Variación anual (%)',
    hovermode='x unified', template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig1.show()

In [ ]:
# ── Gráfica 2: Inflación vs Tasa de política monetaria ────────
# Esta es la gráfica más importante del análisis
# Muestra si Banxico reaccionó a la inflación y con qué rezago
# Usamos dos ejes Y porque las escalas son distintas

fig2 = make_subplots(specs=[[{'secondary_y': True}]])

fig2.add_trace(go.Scatter(
    x=df_mensual.index, y=df_mensual['Inflación general anual'],
    name='Inflación general', line=dict(color='#e63946', width=2.5)
), secondary_y=False)

fig2.add_trace(go.Scatter(
    x=df_mensual.index, y=df_mensual['Tasa objetivo Banxico'],
    name='Tasa objetivo Banxico', line=dict(color='#1d3557', width=2.5)
), secondary_y=True)

fig2.add_hline(y=3, line_dash='dash', line_color='green',
    annotation_text='Meta 3%')

fig2.update_layout(
    title='<b>Inflación vs Tasa de Política Monetaria (2008–2026)</b><br><sup>El Banxico sigue a la inflación con 1-2 meses de rezago — opera una regla de reacción, no de anticipación</sup>',
    hovermode='x unified', template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig2.update_yaxes(title_text='Inflación anual (%)', secondary_y=False)
fig2.update_yaxes(title_text='Tasa objetivo (%)', secondary_y=True)
fig2.show()

In [ ]:
# ── Gráfica 3: Panel de los tres indicadores ──────────────────
# Ver las tres variables juntas ayuda a identificar eventos económicos
# Ejemplo visible: el salto del tipo de cambio en marzo 2020 (pandemia)

fig3 = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Inflación General Anual (%)', 'Tasa Objetivo Banxico (%)', 'Tipo de Cambio USD/MXN'),
    shared_xaxes=True, vertical_spacing=0.08
)

datos_panel = [
    ('Inflación general anual',      '#e63946'),
    ('Tasa objetivo Banxico',         '#1d3557'),
    ('Tipo de cambio USD/MXN (FIX)', '#2a9d8f')
]

for i, (col, color) in enumerate(datos_panel, start=1):
    fig3.add_trace(go.Scatter(
        x=df_mensual.index, y=df_mensual[col],
        name=col, line=dict(color=color, width=2), showlegend=False
    ), row=i, col=1)

fig3.update_layout(
    height=750, template='plotly_white',
    title_text='<b>Panel de Indicadores Macroeconómicos de México (2008–2026)</b><br><sup>Marzo 2020: pandemia — salto del tipo de cambio a 25 pesos | 2022: pico inflacionario post-pandemia</sup>',
    hovermode='x unified'
)
fig3.show()

In [ ]:
# ── Gráfica 4: Matriz de correlación ─────────────────────────
# La correlación mide qué tan relacionadas están dos variables (-1 a 1)
# Importante: la correlación simple no captura rezagos — por eso usaremos Granger después

df_corr = df_mensual.dropna().corr().round(2)
nombres_cortos = ['Inflación\ngeneral', 'Inflación\nsubyacente', 'Tasa\nBanxico', 'USD/MXN']

fig4 = go.Figure(data=go.Heatmap(
    z=df_corr.values, x=nombres_cortos, y=nombres_cortos,
    colorscale='RdBu_r', zmin=-1, zmax=1,
    text=df_corr.values, texttemplate='%{text}', textfont={'size': 14}
))

fig4.update_layout(
    title='<b>Matriz de Correlación de Indicadores Económicos</b><br><sup>La correlación tasa-inflación general es baja porque no captura el rezago de reacción del Banxico</sup>',
    template='plotly_white', width=550, height=450
)
fig4.show()

---
## PARTE 7 — Análisis de Series de Tiempo

### Conceptos clave

**Estacionariedad:** una serie es estacionaria cuando su promedio y varianza no cambian con el tiempo.  
La mayoría de modelos de series de tiempo requieren estacionariedad — hay que verificarla antes de modelar.

**Prueba ADF (Dickey-Fuller Aumentada):**  
- H₀ (hipótesis nula): la serie NO es estacionaria  
- Si p-valor < 0.05 → rechazamos H₀ → la serie SÍ es estacionaria ✅  
- Si p-valor ≥ 0.05 → la serie NO es estacionaria → necesitamos diferenciarla ⚠️

**Diferenciación:** en lugar de analizar el nivel (¿cuánto fue?), analizamos el cambio (¿cuánto cambió?)  
El número de diferenciaciones necesarias es el parámetro **d** en el modelo ARIMA(p, d, q).

In [ ]:
# Extraemos la serie de inflación para trabajar con ella
inflacion = df_mensual['Inflación general anual'].dropna()

print(f'Serie de inflación: {len(inflacion)} observaciones mensuales')
print(f'Periodo: {inflacion.index[0].date()} → {inflacion.index[-1].date()}')
print(f'\nMínimo: {inflacion.min():.2f}% | Máximo: {inflacion.max():.2f}% | Promedio: {inflacion.mean():.2f}%')

In [ ]:
# ── Prueba de estacionariedad ADF ─────────────────────────────
resultado_adf = adfuller(inflacion, autolag='AIC')

print('=' * 55)
print('   PRUEBA DE DICKEY-FULLER AUMENTADA (ADF)')
print('   Variable: Inflación general anual')
print('=' * 55)
print(f'   Estadístico ADF : {resultado_adf[0]:.4f}')
print(f'   p-valor         : {resultado_adf[1]:.4f}')
print(f'   Lags usados     : {resultado_adf[2]}')
print('   Valores críticos:')
for nivel, valor in resultado_adf[4].items():
    print(f'      {nivel}: {valor:.4f}')
print('-' * 55)

if resultado_adf[1] < 0.05:
    print('   RESULTADO: ✅ Serie ESTACIONARIA (p < 0.05)')
else:
    print('   RESULTADO: ⚠️  Serie NO ESTACIONARIA (p ≥ 0.05)')
    print('   Necesitamos diferenciarla antes de modelar')
print('=' * 55)

In [ ]:
# ── Diferenciación ────────────────────────────────────────────
# .diff() calcula la diferencia respecto al mes anterior
# .dropna() elimina el primer valor que queda vacío después de diferenciar
inflacion_diff = inflacion.diff().dropna()

# Segunda diferenciación si la primera no fue suficiente
inflacion_diff2 = inflacion_diff.diff().dropna()

# Probamos estacionariedad en ambas versiones diferenciadas
adf_d1 = adfuller(inflacion_diff,  autolag='AIC')
adf_d2 = adfuller(inflacion_diff2, autolag='AIC')

print('Resultados de estacionariedad:')
print(f'   Serie original    : p = {resultado_adf[1]:.4f} → {"✅ Estacionaria" if resultado_adf[1] < 0.05 else "⚠️  No estacionaria"}')
print(f'   1 diferenciación  : p = {adf_d1[1]:.4f} → {"✅ Estacionaria" if adf_d1[1] < 0.05 else "⚠️  No estacionaria"}')
print(f'   2 diferenciaciones: p = {adf_d2[1]:.4f} → {"✅ Estacionaria" if adf_d2[1] < 0.05 else "⚠️  No estacionaria"}')

# Graficamos la transformación
fig5 = make_subplots(rows=2, cols=1,
    subplot_titles=('Inflación original — NO estacionaria (tiene tendencia)',
                    'Inflación diferenciada — cambio mensual en puntos porcentuales'))
fig5.add_trace(go.Scatter(x=inflacion.index, y=inflacion, line=dict(color='#e63946')), row=1, col=1)
fig5.add_trace(go.Scatter(x=inflacion_diff.index, y=inflacion_diff, line=dict(color='#1d3557')), row=2, col=1)
fig5.update_layout(height=500, template='plotly_white', showlegend=False,
    title='<b>Transformación para Estacionariedad</b>')
fig5.show()

---
## PARTE 8 — Causalidad de Granger

Antes de construir el modelo multivariado (VAR), verificamos las relaciones entre variables.

**¿Qué es la Causalidad de Granger?**  
No es causalidad filosófica — es causalidad **predictiva**.  
"X Granger-causa a Y" significa que saber el historial de X mejora la predicción de Y, comparado con solo usar el historial de Y.

**Pregunta central:** ¿La tasa Banxico predice la inflación, o es la inflación la que predice la tasa?

In [ ]:
# Preparamos datos para las pruebas de Granger
df_granger = df_mensual[['Inflación general anual', 'Tasa objetivo Banxico']].dropna()

# ── Dirección 1: ¿La tasa predice la inflación? ───────────────
res1 = grangercausalitytests(df_granger[['Inflación general anual', 'Tasa objetivo Banxico']], maxlag=6, verbose=False)
print('¿La tasa Banxico Granger-causa la inflación?')
print('-' * 48)
for lag, res in res1.items():
    pval = res[0]['ssr_ftest'][1]
    sig  = '✅ Sí' if pval < 0.05 else '❌ No'
    print(f'   Lag {lag}: p-valor = {pval:.4f} → {sig}')

print()

# ── Dirección 2: ¿La inflación predice la tasa? ───────────────
res2 = grangercausalitytests(df_granger[['Tasa objetivo Banxico', 'Inflación general anual']], maxlag=6, verbose=False)
print('¿La inflación Granger-causa la tasa Banxico?')
print('-' * 48)
for lag, res in res2.items():
    pval = res[0]['ssr_ftest'][1]
    sig  = '✅ Sí' if pval < 0.05 else '❌ No'
    print(f'   Lag {lag}: p-valor = {pval:.4f} → {sig}')

print()
print('💡 Interpretación:')
print('   La causalidad va en una sola dirección: inflación → tasa')
print('   El Banxico REACCIONA a la inflación pasada (lags 1-2 meses)')
print('   No es el Banxico el que anticipa — es la inflación la que lo mueve')

---
## PARTE 9 — Modelos de Proyección

Probamos tres modelos en orden de complejidad creciente:

| Modelo | Variables | Resultado |
|---|---|---|
| ARIMA | Solo inflación | Proyecta línea plana — insuficiente |
| Prophet | Solo inflación | Captura tendencia pero no shocks exógenos |
| VAR | Inflación + tasa + tipo de cambio | El más adecuado — captura interdependencias |

### ¿Por qué ARIMA y Prophet no son suficientes?
Son modelos **univariados** — solo usan la inflación para predecir la inflación.  
Como intentar predecir el clima mirando solo la temperatura, ignorando humedad, viento y presión.  
Los shocks externos (pandemia, guerra, aranceles) no están en los datos históricos de inflación.

In [ ]:
# ── Modelo 1: Auto ARIMA ──────────────────────────────────────
# Divide los datos: 85% para entrenar, 15% para evaluar
# Esto se llama 'train-test split' — el modelo no ve los datos de prueba
n_train = int(len(inflacion) * 0.85)
train   = inflacion[:n_train]
test    = inflacion[n_train:]

print(f'Entrenamiento: {len(train)} meses ({train.index[0].date()} → {train.index[-1].date()})')
print(f'Prueba:        {len(test)} meses  ({test.index[0].date()} → {test.index[-1].date()})')
print('\n⏳ Buscando el mejor modelo ARIMA...')

# auto_arima prueba muchas combinaciones de (p,d,q) y elige la de menor AIC
# AIC = criterio que premia buen ajuste pero penaliza complejidad innecesaria
modelo_arima = pm.auto_arima(
    train, start_p=0, max_p=5, start_q=0, max_q=5,
    d=None, seasonal=False, stepwise=True,
    information_criterion='aic', suppress_warnings=True
)

print(f'\n✅ Mejor modelo: ARIMA{modelo_arima.order}')
print(f'   AIC: {modelo_arima.aic():.2f}')
print(f'\n💡 ARIMA(0,1,1) es equivalente a suavizamiento exponencial')
print('   Proyecta el último valor ajustado hacia adelante — línea plana')

In [ ]:
# Evaluación del modelo ARIMA en el periodo de prueba
pred_test, ci_test = modelo_arima.predict(n_periods=len(test), return_conf_int=True)

mae  = np.mean(np.abs(test.values - pred_test))
rmse = np.sqrt(np.mean((test.values - pred_test)**2))
mape = np.mean(np.abs((test.values - pred_test) / test.values)) * 100

print('=' * 50)
print(f'   EVALUACIÓN ARIMA{modelo_arima.order} en periodo de prueba')
print('=' * 50)
print(f'   MAE  : {mae:.3f} pp — error promedio en puntos porcentuales')
print(f'   RMSE : {rmse:.3f} pp — penaliza más los errores grandes')
print(f'   MAPE : {mape:.2f}%  — error promedio como % del valor real')
print('=' * 50)

In [ ]:
# ── Modelo 2: Prophet ─────────────────────────────────────────
# Prophet descompone la serie en: Tendencia + Estacionalidad + Ruido
# Requiere columnas con nombres exactos: 'ds' (fecha) y 'y' (valor)

df_prophet = df_mensual[['Inflación general anual']].reset_index()
df_prophet.columns = ['ds', 'y']
df_prophet = df_prophet.dropna()

modelo_prophet = Prophet(
    changepoint_prior_scale=0.05,  # Flexibilidad de la tendencia (0.05 = moderada)
    yearly_seasonality=True,       # Busca patrones que se repiten cada año
    weekly_seasonality=False,      # No aplica — datos mensuales
    daily_seasonality=False        # No aplica — datos mensuales
)
modelo_prophet.fit(df_prophet)

# Proyección a 12 meses
futuro_prophet   = modelo_prophet.make_future_dataframe(periods=12, freq='ME')
proyeccion_prophet = modelo_prophet.predict(futuro_prophet)

# Gráfica Prophet
fig_prophet = go.Figure()
fig_prophet.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'],
    name='Inflación real', line=dict(color='#1d3557', width=2)))
fig_prophet.add_trace(go.Scatter(x=proyeccion_prophet['ds'], y=proyeccion_prophet['yhat'],
    name='Proyección Prophet', line=dict(color='#e63946', width=2.5, dash='dash')))
fig_prophet.add_trace(go.Scatter(
    x=list(proyeccion_prophet['ds']) + list(proyeccion_prophet['ds'][::-1]),
    y=list(proyeccion_prophet['yhat_upper']) + list(proyeccion_prophet['yhat_lower'][::-1]),
    fill='toself', fillcolor='rgba(230,57,70,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='IC 95%'))
fig_prophet.add_hline(y=3, line_dash='dash', line_color='green', annotation_text='Meta 3%')
fig_prophet.update_layout(
    title='<b>Proyección Prophet</b><br><sup>⚠️ La inflación real se sale del intervalo en 2009, 2017 y 2022 — shocks exógenos que ningún modelo estadístico puede anticipar</sup>',
    xaxis_title='Fecha', yaxis_title='Inflación anual (%)',
    hovermode='x unified', template='plotly_white', legend=dict(x=0.01, y=0.99)
)
fig_prophet.show()

print('\n💡 Limitación de Prophet en este contexto:')
print('   Los shocks exógenos (crisis 2008, pandemia 2020, guerra Ucrania 2022)')
print('   rompen cualquier patrón histórico — Prophet los ve como anomalías, no los predice')
print('   → Necesitamos un modelo que use más variables: VAR')

In [ ]:
# ── Modelo 3: VAR (Vector AutoRegresivo) ──────────────────────
# A diferencia de ARIMA y Prophet, el VAR es MULTIVARIADO:
# cada variable se explica con el pasado de TODAS las variables simultáneamente
#
# inflación hoy = f(inflación ayer, tasa ayer, dólar ayer)
# tasa hoy      = f(tasa ayer, inflación ayer, dólar ayer)
# dólar hoy     = f(dólar ayer, inflación ayer, tasa ayer)

df_var = df_mensual[[
    'Inflación general anual',
    'Tasa objetivo Banxico',
    'Tipo de cambio USD/MXN (FIX)'
]].dropna()

# Selección del número óptimo de lags usando el criterio AIC
# Lag = cuántos meses hacia atrás mira el modelo
modelo_var = VAR(df_var)
seleccion  = modelo_var.select_order(maxlags=12)
print('Selección de lags óptimos (AIC menor = mejor):')
print(seleccion.summary())

In [ ]:
# Entrenamos el VAR con 3 lags (el óptimo según AIC)
var_resultado = modelo_var.fit(3)

# Proyección a 12 meses
# El VAR necesita los últimos 'p' valores observados como punto de partida
proyeccion_var = var_resultado.forecast(df_var.values[-3:], steps=12)

fechas_var = pd.date_range(
    start=df_var.index[-1] + pd.DateOffset(months=1),
    periods=12, freq='ME'
)

df_proy_var = pd.DataFrame(
    proyeccion_var,
    index=fechas_var,
    columns=df_var.columns
)

print('📅 Proyección VAR — próximos 12 meses')
print('=' * 65)
print(df_proy_var.round(2).to_string())
print('=' * 65)

In [ ]:
# Gráfica VAR — histórico + proyección con banda de incertidumbre
mse          = var_resultado.mse(steps=12)
std_inflacion = np.sqrt([mse[i][0,0] for i in range(12)])
std_tasa      = np.sqrt([mse[i][1,1] for i in range(12)])

fig_var = make_subplots(rows=2, cols=1,
    subplot_titles=(
        'Inflación General Anual — Histórico + Proyección VAR',
        'Tasa Objetivo Banxico — Histórico + Proyección VAR'
    ),
    shared_xaxes=True, vertical_spacing=0.12
)

for row, col, std, color in [
    (1, 'Inflación general anual', std_inflacion, '#e63946'),
    (2, 'Tasa objetivo Banxico',   std_tasa,      '#1d3557')
]:
    fig_var.add_trace(go.Scatter(
        x=df_var.index, y=df_var[col],
        name=f'{col} (real)', line=dict(color=color, width=2)
    ), row=row, col=1)

    fig_var.add_trace(go.Scatter(
        x=fechas_var, y=df_proy_var[col],
        name=f'{col} (VAR)', line=dict(color=color, width=2.5, dash='dash')
    ), row=row, col=1)

    # Banda de incertidumbre al 95% — se amplía con el tiempo (más lejano = más incierto)
    r, g, b = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fig_var.add_trace(go.Scatter(
        x=list(fechas_var) + list(fechas_var[::-1]),
        y=list(df_proy_var[col] + 1.96*std) + list((df_proy_var[col] - 1.96*std)[::-1]),
        fill='toself', fillcolor=f'rgba({r},{g},{b},0.15)',
        line=dict(color='rgba(0,0,0,0)'), showlegend=False
    ), row=row, col=1)

fig_var.add_hline(y=3, line_dash='dash', line_color='green',
    annotation_text='Meta 3%', row=1, col=1)

fig_var.update_layout(
    height=650, template='plotly_white',
    title='<b>Proyección VAR(3) — Política Monetaria México 2026-2027</b><br><sup>La banda de incertidumbre se amplía con el tiempo — entre más lejana la predicción, mayor la incertidumbre</sup>',
    hovermode='x unified'
)
fig_var.show()

---
## PARTE 10 — Hallazgos y Conclusiones

### ¿Qué encontramos?

**1. La inflación mexicana estuvo dominada por shocks externos**  
Crisis financiera 2008-2009, pandemia 2020, y ciclo inflacionario 2021-2022 fueron eventos que ningún modelo estadístico puede anticipar. Quedaron como anomalías en los datos.

**2. El Banco de México opera una regla de reacción, no de anticipación**  
La causalidad de Granger confirma que la inflación predice la tasa (lags 1-2, p<0.05), pero no al revés.  
EL Banco de México mira la inflación de los últimos 1-2 meses y ajusta su tasa en consecuencia.

**3. El VAR es el modelo más adecuado para este análisis**  
Al incorporar inflación, tasa y tipo de cambio simultáneamente, captura las interdependencias reales de la economía. ARIMA y Prophet son insuficientes por ser univariados.

**4. Proyección para 2026**  
El VAR anticipa inflación estabilizándose cerca de 4.25% y tasa bajando hacia ~6% — ciclo de recortes sostenido, pero inflación aún por encima de la meta del 3%.

### Limitaciones del modelo
- No incluye variables como precio del petróleo, IGAE, o expectativas de inflación
- Los shocks exógenos (geopolítica, aranceles) no están en los datos históricos
- El tipo de cambio proyectado (~17.27) puede estar subestimado dado el contexto geopolítico actual

### ¿Qué sigue? (Versión 2.0)
- Agregar precio del petróleo (WTI) como variable exógena
- Incorporar el IGAE (actividad económica mensual) como proxy del PIB
- Agregar expectativas de inflación de Banxico
- Publicar el dashboard en Streamlit Cloud

---
## Resumen de herramientas y conceptos usados

| Tema | Concepto | Herramienta Python |
|---|---|---|
| APIs | Conexión a fuentes de datos oficiales | `sie-banxico` |
| Limpieza de datos | JSON → DataFrame, resampleo mensual | `pandas` |
| Visualización | Gráficas interactivas | `plotly` |
| Estacionariedad | Prueba ADF y diferenciación | `statsmodels` |
| Causalidad | Prueba de Granger | `statsmodels` |
| Modelado univariado | ARIMA automático | `pmdarima` |
| Modelado con tendencia | Descomposición tendencia+estacionalidad | `prophet` |
| Modelado multivariado | VAR — variables interdependientes | `statsmodels` |
| Evaluación | MAE, RMSE, MAPE | `numpy` |